# Upgaming: real pet segmentation experiments
Run in an interactive **free Colab GPU** runtime. Select **Runtime → Change runtime type → T4 GPU** (or another free GPU offered). Availability and session lifetime are not guaranteed. This notebook runs the repository package and does not run the local CPU benchmark or final test evaluation.

Create the source ZIP locally with `python scripts/package_source.py`. Upload `artifacts/upgaming-petseg-source.zip` in Colab's Files sidebar. Keep that source ZIP locally. Do not upload credentials. The default is the full-data A/B/C comparison on CUDA. If free compute is restricted, explicitly select the reduced family before any final run: 1,000 training images, full validation/test, batch size 2. The reduced family is valid on GPU too; its config filenames retain the `cpu_` prefix for compatibility.

Run the cells in order. Stop at the review gate, inspect the data and diagnostic outputs, then record your own review. See `docs/COLAB_HANDOFF.md` for the exact return files and local continuation.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

DEVICE = 'cuda'                 # Change explicitly to 'cpu' only if choosing CPU training.
CONFIG_FAMILY = 'full'        # 'reduced': 1,000 train images; 'full': all training images.
BATCH_SIZE = None               # None preserves the family default; apply any override to A/B/C together.
RUN_NAME = 'full-v1'          # New name if changing source, configs, family, or batch size.
SOURCE_ZIP = Path('/content/upgaming-petseg-source.zip')
PROJECT = Path('/content/upgaming-petseg')
assert DEVICE in {'cuda', 'cpu'} and CONFIG_FAMILY in {'reduced', 'full'}
assert RUN_NAME and all(c.isalnum() or c in '-_' for c in RUN_NAME), 'Use a simple run name'
assert SOURCE_ZIP.is_file(), 'Upload the source ZIP in the Files sidebar first'

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

SOURCE_SHA256 = file_sha256(SOURCE_ZIP)
marker = PROJECT / '.source_zip_sha256'
if marker.exists():
    assert marker.read_text().strip() == SOURCE_SHA256, 'Source changed: use a fresh PROJECT directory and RUN_NAME'
else:
    assert not PROJECT.exists() or not any(PROJECT.iterdir()), 'Choose an empty PROJECT directory'
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ZIP) as archive:
        for info in archive.infolist():
            assert (PROJECT / info.filename).resolve().is_relative_to(PROJECT.resolve()), 'Unsafe ZIP member'
        archive.extractall(PROJECT)
    assert (PROJECT / 'pyproject.toml').is_file(), 'ZIP must contain pyproject.toml at its root'
    marker.write_text(SOURCE_SHA256 + '\n')
os.chdir(PROJECT)
print('Source SHA-256:', SOURCE_SHA256)


## Check the actual training device and preserve the installed torch pair
The next cell runs LR-ASPP on the requested device with random weights, so it downloads no model. A CUDA availability flag alone is insufficient. If it fails, reset/select a GPU runtime and inspect the error. Do not blindly upgrade torch/torchvision. Consult the [official matched pairs](https://pytorch.org/get-started/previous-versions/) if the runtime itself is broken.


In [ ]:
import torch, torchvision
from torchvision.models.segmentation import lraspp_mobilenet_v3_large

TORCH_PAIR = (str(torch.__version__), str(torchvision.__version__))
print(sys.version)
print('torch / torchvision:', TORCH_PAIR)
print('Requested device:', DEVICE, '| CUDA available:', torch.cuda.is_available())
if DEVICE == 'cuda':
    assert torch.cuda.is_available(), 'Select a free GPU runtime, or explicitly choose DEVICE=cpu'
    print('GPU:', torch.cuda.get_device_name(0))

def check_forward():
    model = lraspp_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=2).to(DEVICE).eval()
    with torch.inference_mode():
        result = model(torch.zeros(1, 3, 64, 64, device=DEVICE))['out']
        assert result.shape == (1, 2, 64, 64) and torch.isfinite(result).all()
        assert result.device.type == DEVICE
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
    del result, model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

check_forward()
# Constraints also protect the installed pair from transitive dependency changes.
constraints = Path('/content/petseg-torch-constraints.txt')
constraints.write_text(f'torch=={TORCH_PAIR[0]}\ntorchvision=={TORCH_PAIR[1]}\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
                'numpy>=1.26,<3', 'Pillow>=10.4,<13', 'matplotlib>=3.9,<4',
                'streamlit>=1.40,<2', 'pytest>=8,<10'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', '.'], check=True)
from importlib.metadata import version
assert (version('torch'), version('torchvision')) == TORCH_PAIR, 'Runtime libraries changed; restart and investigate'
helper_modules = {'numpy': 'numpy', 'Pillow': 'PIL', 'matplotlib': 'matplotlib',
                  'streamlit': 'streamlit', 'pytest': 'pytest'}
changed_helpers = [package for package, module in helper_modules.items()
                   if module in sys.modules and getattr(sys.modules[module], '__version__', None) != version(package)]
if changed_helpers:
    raise RuntimeError(f'Installed versions differ from loaded modules: {changed_helpers}. '
                       'Use Runtime → Restart session, then rerun from the first cell.')
check_forward()
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## Verify durable checkpoint storage before training
The default mounts your own Google Drive and writes to a separate named comparison folder. Follow Google's account authorization yourself. The cell verifies a file can be written and read, then records the source/config/split identity. A mounted Drive reduces session-loss risk; it is still necessary to download the result ZIP.

Ephemeral storage requires the deliberate settings `USE_DRIVE=False` and `ALLOW_EPHEMERAL=True`. Files under `/content` can disappear when the runtime ends. Dataset extraction and downloaded pretrained weights remain ephemeral to avoid filling Drive with caches.


In [ ]:
USE_DRIVE = True
ALLOW_EPHEMERAL = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.ismount('/content/drive'), 'Drive is not mounted; do not train yet'
    PERSIST = Path('/content/drive/MyDrive/upgaming-petseg') / RUN_NAME / 'artifacts'
else:
    assert ALLOW_EPHEMERAL, 'Enable ALLOW_EPHEMERAL explicitly to accept loss on session termination'
    PERSIST = Path('/content/upgaming-ephemeral') / RUN_NAME / 'artifacts'
PERSIST.mkdir(parents=True, exist_ok=True)
probe = PERSIST / '.write-read-check'
probe.write_text(SOURCE_SHA256)
assert probe.read_text() == SOURCE_SHA256, 'Persistent storage read/write check failed'
probe.unlink()
DATA = Path('/content/pet-data')

from petseg.utils import environment, read_json, sha256, write_json
from petseg.train import inspect_run, load_config

def run(*args):
    subprocess.run([sys.executable, '-m', 'petseg', *map(str, args)], check=True)

run('env')
AUDIT = Path('/content/pet-audit')
run('prepare', '--root', DATA, '--output', AUDIT, '--download')
MANIFEST = PERSIST / 'data/splits.json'
if (PERSIST / 'data').exists():
    assert MANIFEST.is_file() and sha256(MANIFEST) == sha256(AUDIT / 'splits.json'), 'Stored and current splits differ; preserve the old artifacts and use a new RUN_NAME'
else:
    shutil.copytree(AUDIT, PERSIST / 'data')
prefix = 'cpu_' if CONFIG_FAMILY == 'reduced' else ''
configs = {name: load_config(f'configs/{prefix}{name}.json') for name in ['A', 'B', 'C']}
if BATCH_SIZE is not None:
    assert isinstance(BATCH_SIZE, int) and BATCH_SIZE > 0
    for config in configs.values():
        config['batch_size'] = BATCH_SIZE
comparison = [{k: v for k, v in config.items() if k not in {'experiment', 'frozen', 'augment'}}
              for config in configs.values()]
assert comparison[0] == comparison[1] == comparison[2], 'A/B/C budgets differ'
identity = {'source_zip_sha256': SOURCE_SHA256, 'manifest_sha256': sha256(MANIFEST),
            'family': CONFIG_FAMILY, 'configs': configs, 'device': DEVICE,
            'torch': TORCH_PAIR[0], 'torchvision': TORCH_PAIR[1]}
identity_path = PERSIST / 'workflow/identity.json'
if identity_path.exists():
    assert read_json(identity_path) == identity, 'This run name has different source/configs/splits/runtime; choose a new RUN_NAME'
else:
    write_json(identity_path, identity)
for name, config in configs.items():
    write_json(PERSIST / f'workflow/configs/{name}.json', config)
write_json(PERSIST / 'workflow/runtime.json', environment())
print('Verified output folder:', PERSIST)
print('Split SHA-256:', sha256(MANIFEST))
print('Family:', CONFIG_FAMILY, '| train cap:', configs['A']['train_cap'], '| batch:', configs['A']['batch_size'])
print(read_json(PERSIST / 'data/audit.json')['counts'])
print((PERSIST / 'data/ANNOTATION_README.txt').read_text())
from IPython.display import display, Image as DisplayImage

def show(path):
    display(DisplayImage(filename=str(path)))

show(PERSIST / 'data/audit_examples.png')


## Run and inspect the real-data diagnostics
Raw trimap labels are **1=pet, 2=background, 3=ambiguous outline**; the project maps these to **1, 0, 255**. Yellow audit pixels are ignored outlines. Check photograph/target alignment and read the actual annotation README above.

The overfit check learns two training images; its displayed IoU is measured on those same training images and says nothing about held-out quality. The smoke run checks full-network training. Both remain outside final A/B/C selection.


In [ ]:
import math
from petseg.data import stratified_cap
from petseg.evaluate import evaluate_model, save_evaluation
from petseg.inference import load_model

def train_or_resume(config_path, output):
    config = load_config(config_path)
    if (output / 'latest.pt').exists():
        checked = inspect_run(output, config=config, manifest=MANIFEST)
        if checked['completed']:
            print('Validated completed run:', output.name)
            return checked
        resume = ['--resume', checked['resume']]
    else:
        assert not output.exists() or not any(output.iterdir()), f'{output}: files without latest.pt; preserve them and choose a new run name'
        resume = []
    run('train', '--config', config_path, '--root', DATA, '--manifest', MANIFEST,
        '--output', output, '--device', DEVICE, *resume)
    return inspect_run(output, config=config, manifest=MANIFEST, require_completed=True)

for diagnostic in ['overfit', 'smoke']:
    output = PERSIST / 'diagnostics' / diagnostic
    checked = train_or_resume(Path(f'configs/{diagnostic}.json'), output)
    history = checked['latest']['history']
    assert history and all(math.isfinite(row['train_loss']) for row in history), 'Non-finite diagnostic loss'
    print(diagnostic, '| first loss:', history[0]['train_loss'], '| last loss:', history[-1]['train_loss'])
    checkpoint = output / ('latest.pt' if diagnostic == 'overfit' else 'best.pt')
    model, state = load_model(checkpoint, DEVICE)
    if diagnostic == 'overfit':
        records = stratified_cap(read_json(MANIFEST)['splits']['train'], 2, state['config']['seed'])
        label = 'same_two_training_images'
        assert history[-1]['train_loss'] < history[0]['train_loss'], 'Overfit loss did not decrease: investigate before A/B/C'
    else:
        records = read_json(MANIFEST)['splits']['val'][:2]
        label = 'two_validation_images_diagnostic_only'
    metrics, rows = evaluate_model(model, DATA, records, state['config']['size'], DEVICE)
    metrics.update({'diagnostic_only': True, 'scope': label, 'checkpoint_sha256': sha256(checkpoint)})
    example_output = output / label
    save_evaluation(example_output, metrics, rows, model, DATA, records, DEVICE)
    print(label, 'pet IoU:', metrics['foreground_iou'])
    show(example_output / 'ranked_examples.png')
    del model


## Review gate: edit only after inspecting the outputs
Before enabling the next cell, check the audit target corresponds to the pet, ignored boundaries are yellow, overfit loss drops substantially, the two training predictions follow their masks, and the smoke run is finite and completed. If the checks fail, keep the flags false, download the diagnostic artifacts, and investigate.

These flags record the notebook operator's own review. Running the notebook does not establish that the candidate understands the implementation.


In [ ]:
AUDIT_REVIEWED = False
DIAGNOSTICS_REVIEWED = False
assert AUDIT_REVIEWED and DIAGNOSTICS_REVIEWED, 'Inspect the images and losses above, then set the two review flags yourself'
write_json(PERSIST / 'workflow/review.json', {
    'audit_reviewed_by_notebook_operator': AUDIT_REVIEWED,
    'diagnostics_reviewed_by_notebook_operator': DIAGNOSTICS_REVIEWED,
    'manifest_sha256': sha256(MANIFEST),
    'diagnostic_checkpoint_sha256': {'overfit': sha256(PERSIST / 'diagnostics/overfit/latest.pt'),
                                     'smoke': sha256(PERSIST / 'diagnostics/smoke/best.pt')},
    'recorded_at_utc': datetime.now(timezone.utc).isoformat()})


## Complete A/B/C and select using validation only
All three experiments start independently; C does not continue B. Every run uses the same saved family, batch size, seed and split. Completed runs are validated against their checkpoints, hashes, configs, histories and manifest before being skipped. Interrupted runs resume only from their own complete run folder at an epoch boundary.

If memory runs out, use a **new RUN_NAME**, apply a common smaller `BATCH_SIZE` to all three experiments, and restart the comparison. Keep old evidence. Never edit one experiment or mix full/reduced families in the same comparison. On a later Colab session, upload the same source ZIP, use the same settings and run cells from the start to remount Drive and re-extract data. Runtime/library changes require a new comparison folder.


In [ ]:
assert AUDIT_REVIEWED and DIAGNOSTICS_REVIEWED, 'Complete the review gate first'
assert read_json(PERSIST / 'workflow/identity.json') == identity
for experiment in ['A', 'B', 'C']:
    train_or_resume(PERSIST / f'workflow/configs/{experiment}.json', PERSIST / 'runs' / experiment)
run('select', '--runs', PERSIST / 'runs', '--manifest', MANIFEST, '--output', PERSIST / 'final')
run('report', '--artifacts', PERSIST)
print((PERSIST / 'reports/results.md').read_text())
print('Selection SHA-256:', sha256(PERSIST / 'final/selected.pt'))


## Download the artifacts and their integrity records
Run this export cell after training. You can also run it after an interruption to return partial evidence; the inventory records completion or validation errors and does not label incomplete work as final. Stop any training cell before exporting so files cannot change during ZIP creation.

The ZIP always contains a top-level `artifacts/` folder. It includes complete best/latest run folders, diagnostics, manifests, selected weights and reports; dataset files and caches are excluded. Checkpoints can make the ZIP several hundred MB. Download both the ZIP and its `.sha256` file and keep the saved notebook output. [Colab FAQ](https://research.google.com/colaboratory/faq.html) describes runtime persistence and free resource limits.


In [ ]:
# Run only when no training process is writing checkpoints.
completion = {}
for name in ['A', 'B', 'C']:
    output = PERSIST / 'runs' / name
    if not (output / 'latest.pt').exists():
        completion[name] = {'completed': False, 'reason': 'no epoch checkpoint'}
        continue
    try:
        checked = inspect_run(output, config=configs[name], manifest=MANIFEST)
        completion[name] = {'completed': checked['completed']}
    except (ValueError, FileNotFoundError, RuntimeError) as exc:
        completion[name] = {'completed': False, 'validation_error': str(exc)}

inventory_path = PERSIST / 'transfer_inventory.json'
paths = sorted(path for path in PERSIST.rglob('*')
               if path.is_file() and path != inventory_path
               and 'cache' not in path.relative_to(PERSIST).parts and not path.name.endswith('.tmp'))
write_json(inventory_path, {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_zip_sha256': SOURCE_SHA256, 'manifest_sha256': sha256(MANIFEST),
    'config_family': CONFIG_FAMILY, 'runs': completion,
    'files': [{'path': 'artifacts/' + path.relative_to(PERSIST).as_posix(),
               'bytes': path.stat().st_size, 'sha256': sha256(path)} for path in paths]})
archive_path = Path('/content') / f'upgaming-petseg-{RUN_NAME}-artifacts.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [*paths, inventory_path]:
        archive.write(path, 'artifacts/' + path.relative_to(PERSIST).as_posix())
with zipfile.ZipFile(archive_path) as archive:
    assert archive.testzip() is None, 'ZIP integrity check failed'
archive_hash = sha256(archive_path)
sidecar = archive_path.with_suffix('.zip.sha256')
sidecar.write_text(archive_hash + '  ' + archive_path.name + '\n')
print('Run completion:', completion)
print('Download:', archive_path.name, '| bytes:', archive_path.stat().st_size)
print('ZIP SHA-256:', archive_hash)
from google.colab import files
files.download(str(archive_path))
files.download(str(sidecar))


Return the ZIP, its hash sidecar, and the notebook saved with actual outputs. If upload limits prevent returning a large ZIP, retain the complete Drive folder and return its `transfer_inventory.json`, `workflow/`, `data/`, all A/B/C `status.json`, `config.json`, `environment.json`, `training_ids.json`, `history.json`/CSV, checkpoint hash sidecars, and `final/selection.json`. Local benchmarking still requires the actual selected checkpoint and its hash; do not provide a public link without choosing to publish it yourself.

On the local CPU machine, verify the ZIP hash and inventory, extract into the repository, recreate data in a separate audit folder and compare the manifest hash, then run the CPU benchmark, frozen final test evaluation, and report commands in `docs/COLAB_HANDOFF.md`. No final test scores or CPU optimization result are produced by this training notebook.
